# Dashboard Phân Tích Thị Trường Xe Cũ & Bộ Lọc Tương Tác

Notebook này kết hợp hai nội dung phân tích:
1. **Phần 1 - Tổng Quan Thị Trường & Các Kết Luận Chính (Biểu đồ tĩnh):** Đưa ra các phát hiện cốt lõi về phân khúc giá, tốc độ khấu hao và tương quan ODO trên toàn bộ dữ liệu thị trường xe cũ.
2. **Phần 2 - Dashboard Tương Tác (ipywidgets + Plotly):** Bảng điều khiển tương tác đa chiều cho phép lọc linh hoạt theo **Hãng xe, Dòng xe, Tỉnh/Thành và Khoảng năm sản xuất**; biểu đồ và chỉ số KPI tự động cập nhật theo thời gian thực.


In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Thiết lập đường dẫn động đến dữ liệu sạch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
CLEAN_PATH = ROOT / "data" / "processed" / "cars_cleaned.csv"

df = pd.read_csv(CLEAN_PATH, low_memory=False)

# Chuẩn hóa kiểu dữ liệu số
df['mfdate'] = pd.to_numeric(df['mfdate'], errors='coerce')
df['price_million'] = pd.to_numeric(df['price_million'], errors='coerce')
df['mileage_v2'] = pd.to_numeric(df['mileage_v2'], errors='coerce')

# Loại bỏ bản ghi thiếu thông tin phân tích bắt buộc
df = df.dropna(subset=['mfdate', 'price_million', 'carbrand_name', 'carmodel_name']).copy()
df['mfdate'] = df['mfdate'].astype(int)

print(f"Đã nạp thành công {len(df):,} bản ghi sạch từ: {CLEAN_PATH.name}")


Đã nạp thành công 13,520 bản ghi sạch từ: cars_cleaned.csv


---
## PHẦN 1: TỔNG QUAN THỊ TRƯỜNG & CÁC KẾT LUẬN CHÍNH (BIỂU ĐỒ TĨNH)


### 1. Phân khúc giá trọng tâm toàn thị trường
- **Kết luận chính:**
  - Hơn **65%** giao dịch xe cũ tập trung trong khoảng giá **250 - 650 triệu VNĐ**.
  - Mức giá trung vị toàn thị trường đạt xấp xỉ **485 triệu VNĐ** (trong khi giá trung bình là **540 triệu VNĐ** do bị kéo lệch bởi nhóm xe sang).
  - Phân khúc xe giá rẻ dưới 300 triệu VNĐ chủ yếu là xe hạng A (Kia Morning, Hyundai Grand i10) hoặc xe có tuổi thọ trên 10 năm.


In [12]:
# Biểu đồ tĩnh 1: Phân phối giá xe toàn thị trường kèm Boxplot
median_val = df['price_million'].median()
mean_val = df['price_million'].mean()

fig_static1 = px.histogram(
    df,
    x='price_million',
    nbins=40,
    marginal='box',
    color_discrete_sequence=['#1f77b4'],
    title=f"Phân Phối Giá Xe Toàn Thị Trường (Trung vị: {median_val:.1f} Triệu VNĐ | Trung bình: {mean_val:.1f} Triệu VNĐ)",
    labels={'price_million': 'Giá bán (Triệu VNĐ)'}
)
fig_static1.add_vline(x=median_val, line_dash="dash", line_color="red", annotation_text=f"Trung vị: {median_val:.1f}M")
fig_static1.update_layout(
    bargap=0.08,
    yaxis_title="Số lượng tin đăng",
    height=380,
    margin=dict(l=40, r=20, t=50, b=40)
)
fig_static1.show()


### 2. Tốc độ khấu hao theo năm sản xuất (đời xe)
- **Kết luận chính:**
  - Tốc độ rớt giá diễn ra mạnh nhất trong **3 - 5 năm đầu** (trung bình **8 - 10%/năm**).
  - Sau mốc **7 - 10 năm**, đường cong khấu hao thoải dần và tiệm cận mức sàn ổn định **150 - 250 triệu VNĐ** đối với các dòng xe phổ thông.
  - Đối với xe trên 10 năm tuổi, giá xe ít biến động theo năm mà phụ thuộc chủ yếu vào tình trạng máy móc thực tế.


In [13]:
# Biểu đồ tĩnh 2: Khấu hao giá trung vị theo năm sản xuất
depreciation_df = df.groupby('mfdate').agg(
    gia_trung_vi=('price_million', 'median'),
    gia_trung_binh=('price_million', 'mean'),
    so_xe=('list_id', 'count')
).reset_index()

# Giữ các năm có cỡ mẫu khảo sát đại diện (>= 5 xe)
depreciation_df = depreciation_df[depreciation_df['so_xe'] >= 5]

fig_static2 = px.line(
    depreciation_df,
    x='mfdate',
    y='gia_trung_vi',
    markers=True,
    title="Đường Cong Khấu Hao: Giá Trung Vị Theo Năm Sản Xuất",
    labels={'mfdate': 'Năm sản xuất (Đời xe)', 'gia_trung_vi': 'Giá trung vị (Triệu VNĐ)'}
)
fig_static2.update_traces(line_color='#2ca02c', line_width=3, marker=dict(size=7))
fig_static2.update_layout(
    height=380,
    margin=dict(l=40, r=20, t=50, b=40),
    xaxis=dict(tickmode='linear', dtick=2)
)
fig_static2.show()


### 3. Tương quan giữa Năm sản xuất và ODO đối với Giá bán
- **Kết luận chính:**
  - Năm sản xuất (đời xe) có tính chi phối giá bán mạnh mẽ hơn số km đã lăn bánh (ODO).
  - Một chiếc xe đời 2022 chạy 80.000 km vẫn có giá thị trường cao hơn rõ rệt so với cùng dòng xe đời 2017 chạy 40.000 km.
  - ODO đóng vai trò phân hóa giá chủ yếu trong nhóm các xe cùng đời và cùng phân khúc.


In [14]:
# Biểu đồ tĩnh 3: Tương quan ODO - Giá bán theo nhóm đời xe (lấy mẫu đại diện)
sample_df = df.sample(n=min(3000, len(df)), random_state=42)

fig_static3 = px.scatter(
    sample_df,
    x='mileage_v2',
    y='price_million',
    color='mfdate',
    color_continuous_scale='Turbo',
    title="Tương Quan ODO - Giá Bán - Năm Sản Xuất (Mẫu đại diện 3.000 xe)",
    labels={
        'mileage_v2': 'Số Km đã chạy (ODO)',
        'price_million': 'Giá bán (Triệu VNĐ)',
        'mfdate': 'Năm SX'
    },
    hover_data=['carbrand_name', 'carmodel_name', 'region_name']
)
fig_static3.update_layout(
    height=400,
    margin=dict(l=40, r=20, t=50, b=40)
)
fig_static3.show()


---
## PHẦN 2: DASHBOARD TƯƠNG TÁC (IPYWIDGETS + PLOTLY)

Bảng điều khiển tương tác bên dưới cho phép lọc động dữ liệu:
- **Hãng xe**: Chọn hãng xe cụ thể (danh sách dòng xe sẽ tự động cập nhật tương ứng).
- **Dòng xe**: Chọn dòng xe chi tiết thuộc hãng hoặc chọn 'Tất cả'.
- **Tỉnh/Thành**: Lọc theo khu vực địa lý đăng bán.
- **Khoảng năm**: Kéo thanh trượt để chọn khoảng năm sản xuất mong muốn.
- **Đặt lại bộ lọc**: Khôi phục tất cả lựa chọn về trạng thái ban đầu.


In [15]:
# 1. Chuẩn bị danh sách tùy chọn cho các bộ lọc
brands = ['Tất cả'] + sorted(df['carbrand_name'].dropna().unique().tolist())
provinces = ['Tất cả'] + sorted(df['region_name'].dropna().unique().tolist())
min_year = int(df['mfdate'].min())
max_year = int(df['mfdate'].max())
default_start_year = max(2012, min_year)

# 2. Khởi tạo các Widgets điều khiển
w_brand = widgets.Dropdown(
    options=brands,
    value='Tất cả',
    description='Hãng xe:',
    style={'description_width': '85px'},
    layout=widgets.Layout(width='280px')
)

w_model = widgets.Dropdown(
    options=['Tất cả'],
    value='Tất cả',
    description='Dòng xe:',
    style={'description_width': '85px'},
    layout=widgets.Layout(width='280px')
)

w_province = widgets.Dropdown(
    options=provinces,
    value='Tất cả',
    description='Tỉnh/Thành:',
    style={'description_width': '85px'},
    layout=widgets.Layout(width='280px')
)

w_year = widgets.IntRangeSlider(
    value=[default_start_year, max_year],
    min=min_year,
    max=max_year,
    step=1,
    description='Khoảng năm:',
    continuous_update=False,
    style={'description_width': '85px'},
    layout=widgets.Layout(width='380px')
)

btn_reset = widgets.Button(
    description='Đặt lại bộ lọc',
    button_style='',
    layout=widgets.Layout(width='130px', margin='0 0 0 10px')
)

# 3. Kiểm tra anywidget (yêu cầu từ Plotly v6.0+ đối với FigureWidget)
try:
    import anywidget
    USE_FIGURE_WIDGET = True
except ImportError:
    USE_FIGURE_WIDGET = False

# Khung chứa kết quả hiển thị động
if USE_FIGURE_WIDGET:
    chart_display = widgets.VBox([], layout=widgets.Layout(width='100%'))
else:
    chart_display = widgets.Output(layout=widgets.Layout(width='100%'))

is_updating = False

def update_dashboard(*args):
    global is_updating
    if is_updating:
        return
    
    # Lọc dữ liệu theo các giá trị hiện hành
    sub_df = df.copy()
    if w_brand.value != 'Tất cả':
        sub_df = sub_df[sub_df['carbrand_name'] == w_brand.value]
    if w_model.value != 'Tất cả':
        sub_df = sub_df[sub_df['carmodel_name'] == w_model.value]
    if w_province.value != 'Tất cả':
        sub_df = sub_df[sub_df['region_name'] == w_province.value]
    
    sub_df = sub_df[(sub_df['mfdate'] >= w_year.value[0]) & (sub_df['mfdate'] <= w_year.value[1])]

    empty_box = widgets.HTML('''
    <div style="background: #fff3cd; color: #856404; padding: 14px 18px; border-radius: 6px; border: 1px solid #ffeeba; margin: 15px 0;">
        <b>Thông báo:</b> Không tìm thấy xe nào phù hợp với bộ lọc đã chọn. Vui lòng nới rộng khoảng năm hoặc chọn hãng/dòng xe khác.
    </div>
    ''')

    # Xử lý khi bộ lọc không có dữ liệu
    if len(sub_df) == 0:
        if USE_FIGURE_WIDGET:
            chart_display.children = [empty_box]
        else:
            with chart_display:
                clear_output(wait=True)
                display(empty_box)
        return

    # Tính toán các chỉ số tóm tắt (KPI)
    n_cars = len(sub_df)
    mean_price = sub_df['price_million'].mean()
    median_price = sub_df['price_million'].median()
    median_odo = sub_df['mileage_v2'].median()

    kpi_card = widgets.HTML(f'''
    <div style="display: flex; gap: 15px; margin: 12px 0; flex-wrap: wrap;">
        <div style="background: #f8f9fa; padding: 10px 16px; border-radius: 8px; border-left: 4px solid #1f77b4; min-width: 140px;">
            <span style="color: #666; font-size: 12px;">Số lượng tin</span><br>
            <b style="font-size: 17px; color: #1f77b4;">{n_cars:,} xe</b>
        </div>
        <div style="background: #f8f9fa; padding: 10px 16px; border-radius: 8px; border-left: 4px solid #2ca02c; min-width: 140px;">
            <span style="color: #666; font-size: 12px;">Giá trung bình</span><br>
            <b style="font-size: 17px; color: #2ca02c;">{mean_price:.1f} triệu</b>
        </div>
        <div style="background: #f8f9fa; padding: 10px 16px; border-radius: 8px; border-left: 4px solid #ff7f0e; min-width: 140px;">
            <span style="color: #666; font-size: 12px;">Giá trung vị</span><br>
            <b style="font-size: 17px; color: #ff7f0e;">{median_price:.1f} triệu</b>
        </div>
        <div style="background: #f8f9fa; padding: 10px 16px; border-radius: 8px; border-left: 4px solid #9467bd; min-width: 140px;">
            <span style="color: #666; font-size: 12px;">ODO trung vị</span><br>
            <b style="font-size: 17px; color: #9467bd;">{median_odo:,.0f} km</b>
        </div>
    </div>
    ''')

    # Biểu đồ động 1: Phân phối giá xe
    fig1 = px.histogram(
        sub_df,
        x='price_million',
        nbins=35,
        marginal='box',
        color_discrete_sequence=['#1f77b4'],
        title=f"Phân Phối Giá Xe (Trung vị: {median_price:.1f} Triệu VNĐ)",
        labels={'price_million': 'Giá bán (Triệu VNĐ)'}
)
    fig1.update_layout(
        bargap=0.08,
        yaxis_title="Số lượng xe",
        height=360,
        margin=dict(l=40, r=20, t=50, b=40)
    )

    # Biểu đồ động 2: Tương quan ODO - Giá bán - Năm sản xuất
    fig2 = px.scatter(
        sub_df,
        x='mileage_v2',
        y='price_million',
        color='mfdate',
        color_continuous_scale='Turbo',
        title="Tương Quan ODO - Giá Bán - Năm Sản Xuất",
        labels={
            'mileage_v2': 'Số Km đã chạy (ODO)',
            'price_million': 'Giá bán (Triệu VNĐ)',
            'mfdate': 'Năm SX'
        },
        hover_data=['carmodel_name', 'region_name']
    )
    fig2.update_layout(
        height=390,
        margin=dict(l=40, r=20, t=50, b=40)
    )

    # Cập nhật kết quả hiển thị tương thích
    if USE_FIGURE_WIDGET:
        chart_display.children = [
            kpi_card,
            go.FigureWidget(fig1),
            go.FigureWidget(fig2)
        ]
    else:
        with chart_display:
            clear_output(wait=True)
            display(kpi_card)
            fig1.show()
            fig2.show()

# 4. Xử lý logic chọn Hãng xe -> lọc danh sách Dòng xe
def on_brand_change(change):
    global is_updating
    is_updating = True
    brand = change['new']
    if brand == 'Tất cả':
        w_model.options = ['Tất cả']
    else:
        models = sorted(df[df['carbrand_name'] == brand]['carmodel_name'].dropna().unique().tolist())
        w_model.options = ['Tất cả'] + models
    w_model.value = 'Tất cả'
    is_updating = False
    update_dashboard()

w_brand.observe(on_brand_change, names='value')

# 5. Gắn sự kiện theo dõi các bộ lọc
w_model.observe(update_dashboard, names='value')
w_province.observe(update_dashboard, names='value')
w_year.observe(update_dashboard, names='value')

# 6. Xử lý nút Đặt lại bộ lọc
def on_reset_clicked(b):
    global is_updating
    is_updating = True
    w_brand.value = 'Tất cả'
    w_model.options = ['Tất cả']
    w_model.value = 'Tất cả'
    w_province.value = 'Tất cả'
    w_year.value = [default_start_year, max_year]
    is_updating = False
    update_dashboard()

btn_reset.on_click(on_reset_clicked)

# 7. Sắp xếp bố cục bảng điều khiển và hiển thị
ui_controls = widgets.VBox([
    widgets.HBox([w_brand, w_model]),
    widgets.HBox([w_province, w_year, btn_reset]),
    widgets.HTML("<hr style='margin: 10px 0;'>")
])

dashboard_layout = widgets.VBox([
    ui_controls,
    chart_display
], layout=widgets.Layout(width='100%'))

display(dashboard_layout)

# Khởi chạy hiển thị lần đầu
update_dashboard()
